# 05 - Gold Layer: Sales & Product History Analytics

**Project:** Retail Analytics & Product Dimension History

## What this notebook does
Builds business-consumable gold tables. The centerpiece is a POINT-IN-TIME
join between transactions and the SCD Type 2 product dimension - matching
each transaction to the product attributes that were actually true on that
transaction's date, not whatever the product looks like today. This is the
entire reason SCD Type 2 exists, made concrete.

## Tables created
- `main.retail_analytics.silver_transactions_enriched` (point-in-time join)
- `main.retail_analytics.gold_sales_by_category`
- `main.retail_analytics.gold_sales_by_store`
- `main.retail_analytics.gold_customer_summary`
- `main.retail_analytics.gold_price_history_impact`

In [0]:
%sql
CREATE OR REPLACE TABLE main.retail_analytics.silver_transactions_enriched
COMMENT 'Transactions joined to product attributes AS OF the transaction date (point-in-time correct), using the SCD Type 2 dimension. This is different from joining to "current" product data - see gold_price_history_impact for a direct comparison showing why it matters.'
AS
SELECT
  t.transaction_id,
  t.transaction_date,
  t.customer_id,
  t.product_id,
  t.store_id,
  t.quantity,
  t.discount_pct,
  t.payment_method,
  p.product_name,
  p.category,
  p.subcategory,
  p.unit_price AS price_at_time_of_sale,
  p.cost_price AS cost_at_time_of_sale,
  (t.quantity * p.unit_price * (1 - t.discount_pct)) AS revenue
FROM main.retail_analytics.silver_transactions t
JOIN main.retail_analytics.silver_dim_products_scd2 p
  ON t.product_id = p.product_id
  AND t.transaction_date >= p.effective_start_date
  AND (t.transaction_date < p.effective_end_date OR p.effective_end_date IS NULL);

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.retail_analytics.gold_sales_by_category
COMMENT 'Total revenue and transaction volume by product category, using point-in-time correct pricing'
AS
SELECT
  category,
  COUNT(*) AS transaction_count,
  SUM(quantity) AS units_sold,
  ROUND(SUM(revenue), 2) AS total_revenue,
  ROUND(AVG(revenue), 2) AS avg_transaction_revenue
FROM main.retail_analytics.silver_transactions_enriched
GROUP BY category
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.retail_analytics.gold_sales_by_store
COMMENT 'Total revenue and transaction volume by store'
AS
SELECT
  s.store_name,
  s.region,
  COUNT(*) AS transaction_count,
  ROUND(SUM(e.revenue), 2) AS total_revenue
FROM main.retail_analytics.silver_transactions_enriched e
JOIN main.retail_analytics.silver_stores s ON e.store_id = s.store_id
GROUP BY s.store_name, s.region
ORDER BY total_revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.retail_analytics.gold_customer_summary
COMMENT 'Customer purchase behavior summary'
AS
SELECT
  c.customer_id,
  c.city,
  c.age,
  COUNT(e.transaction_id) AS total_transactions,
  ROUND(SUM(e.revenue), 2) AS total_spent,
  ROUND(AVG(e.revenue), 2) AS avg_transaction_value
FROM main.retail_analytics.silver_customers c
LEFT JOIN main.retail_analytics.silver_transactions_enriched e ON c.customer_id = e.customer_id
GROUP BY c.customer_id, c.city, c.age;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.retail_analytics.gold_price_history_impact
COMMENT 'Compares revenue calculated using point-in-time correct pricing (silver_transactions_enriched) vs. a NAIVE join to only the CURRENT product version. Demonstrates why SCD Type 2 matters: a naive current-only join silently loses ALL historical revenue for discontinued products, and misprices historical transactions for changed products.'
AS
WITH naive_current_only_join AS (
  SELECT
    t.transaction_id,
    t.product_id,
    (t.quantity * p.unit_price * (1 - t.discount_pct)) AS naive_revenue
  FROM main.retail_analytics.silver_transactions t
  JOIN main.retail_analytics.silver_dim_products_scd2 p
    ON t.product_id = p.product_id AND p.is_current = true
)
SELECT
  e.product_id,
  e.product_name,
  COUNT(e.transaction_id) AS correct_transaction_count,
  ROUND(SUM(e.revenue), 2) AS correct_point_in_time_revenue,
  COUNT(n.transaction_id) AS naive_transaction_count,
  ROUND(SUM(n.naive_revenue), 2) AS naive_current_price_revenue,
  ROUND(SUM(e.revenue) - COALESCE(SUM(n.naive_revenue), 0), 2) AS revenue_discrepancy
FROM main.retail_analytics.silver_transactions_enriched e
LEFT JOIN naive_current_only_join n ON e.transaction_id = n.transaction_id
WHERE e.product_id IN ('P001', 'P002', 'P003', 'P010', 'P015')
GROUP BY e.product_id, e.product_name
ORDER BY e.product_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM main.retail_analytics.gold_price_history_impact;

product_id,product_name,correct_transaction_count,correct_point_in_time_revenue,naive_transaction_count,naive_current_price_revenue,revenue_discrepancy
P001,Like Camera,102,482608.51,102,530869.65,-48261.14
P002,Audience Television,105,258359.72,105,284196.95,-25837.23
P003,Here Footwear,87,86180.06,87,94797.3,-8617.24
P010,Step Smartphone,107,324442.4,107,324442.4,0.0
P015,Whether Bags,105,80188.42,0,null,80188.42
